# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inshrah-malik/inshrah-flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Load Dataset

In [11]:
from datasets import load_dataset

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train[:10000]"
)

df = ds.to_pandas()

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  624kB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 8.93MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 3.29MB            

...

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  149MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  146MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

In [12]:
df["ctr"] = (
    df["gsc_clicks"] /
    df["gsc_impressions"].replace(0, 1)
)

In [13]:
features = [
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "sessions_organic",
    "sessions_direct",
    "sessions_social",
    "scroll_events"
]

target = "ctr"

model_df = df[features + [target]].dropna()

In [14]:
from sklearn.model_selection import train_test_split

X = model_df[features]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

Baseline Model

In [15]:
import numpy as np
from sklearn.metrics import mean_absolute_error

baseline_prediction = np.repeat(
    y_train.mean(),
    len(y_test)
)

baseline_mae = mean_absolute_error(
    y_test,
    baseline_prediction
)

baseline_mae

0.019303336835830548

Train Linear Regression

In [16]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()

lr.fit(X_train,y_train)

pred_lr = lr.predict(X_test)

In [17]:
from sklearn.metrics import mean_absolute_error

mae_lr = mean_absolute_error(
    y_test,
    pred_lr
)

mae_lr

0.019508072474219908

Decision Tree

In [18]:
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(

    random_state=42,

    max_depth=5

)

tree.fit(X_train,y_train)

pred_tree = tree.predict(X_test)

mae_tree = mean_absolute_error(
    y_test,
    pred_tree
)

mae_tree

0.01809402016668154

Random Forest

In [19]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(

    random_state=42,

    n_estimators=100

)

rf.fit(X_train,y_train)

pred_rf = rf.predict(X_test)

mae_rf = mean_absolute_error(
    y_test,
    pred_rf
)

mae_rf

0.019130905108834476

Compare Models

In [20]:
import pandas as pd

results = pd.DataFrame({

    "Model":[

        "Baseline",

        "Linear Regression",

        "Decision Tree",

        "Random Forest"

    ],

    "MAE":[

        baseline_mae,

        mae_lr,

        mae_tree,

        mae_rf

    ]

})

results

,Model,MAE
0,Baseline,0.019303
1,Linear Regression,0.019508
2,Decision Tree,0.018094
3,Random Forest,0.019131


In [21]:
importance = pd.DataFrame({

    "Feature":features,

    "Importance":rf.feature_importances_

})

importance.sort_values(
    "Importance",
    ascending=False
)

,Feature,Importance
1,gsc_avg_position,0.699065
0,gsc_impressions,0.300935
2,ga4_pageviews,0.000000
3,ga4_sessions,0.000000
4,ga4_users,0.000000
5,sessions_organic,0.000000
6,sessions_direct,0.000000
7,sessions_social,0.000000
8,scroll_events,0.000000


Error Analysis

In [22]:
comparison = pd.DataFrame({

    "Actual":y_test,

    "Predicted":pred_rf

})

comparison.head(20)

,Actual,Predicted
5344,0.000000,0.003128
7444,0.000000,0.000000
1731,0.000000,0.021709
8719,0.000000,0.000000
4521,0.000000,0.019846
7453,0.000000,0.000000
576,0.000000,0.000000
7428,0.000000,0.011250
5577,0.000000,0.002800
439,0.000000,0.019831


In [23]:
comparison["Error"] = abs(comparison["Actual"] - comparison["Predicted"])
comparison.sort_values("Error", ascending=False).head(10)

,Actual,Predicted,Error
4687,1.000000,0.000000,1.000000
3463,1.000000,0.100374,0.899626
6658,1.000000,0.203093,0.796907
1002,0.500000,0.000000,0.500000
5407,0.500000,0.000000,0.500000
1418,0.500000,0.051764,0.448236
7474,0.375000,0.011956,0.363044
393,0.333333,0.000000,0.333333
427,0.333333,0.000000,0.333333
8507,0.333333,0.000000,0.333333


In [24]:
from sklearn.model_selection import GroupShuffleSplit

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*



## Finding 1

The FlyRank paper reports that the proposed model performs better than the baseline model.

### My methodology question

How was the evaluation split created? If pages from the same client appear in both the training and testing sets, the reported performance may be optimistic. A grouped or time-aware split would provide a stronger estimate of generalisation.

---

## Finding 2

The paper shows that historical search performance helps predict future outcomes.

### My methodology question

How were the target labels generated? I would like to confirm that only information available before the prediction time was used, so that no future information leaked into the model.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Section 1 completed.")

Section 1 completed.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*



I compared the original random train/test split with a grouped split based on `client_hash_id`.

The grouped split prevents data from the same client appearing in both training and testing datasets. This provides a more realistic estimate of how the model may perform on unseen clients.

The comparison below shows the evaluation metrics for both validation strategies.

In [26]:
import pandas as pd

In [27]:
df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"]
df["ctr"] = df["ctr"].fillna(0)

In [28]:
features = [
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "scroll_events"
]

X = df[features]
y = df["ctr"]

In [29]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)

before_mae = mean_absolute_error(y_test, pred)
before_mse = mean_squared_error(y_test, pred)
before_r2 = r2_score(y_test, pred)

In [30]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

model = RandomForestRegressor(random_state=42)

model.fit(X_train, y_train)

pred = model.predict(X_test)

after_mae = mean_absolute_error(y_test, pred)
after_mse = mean_squared_error(y_test, pred)
after_r2 = r2_score(y_test, pred)

In [31]:
results = pd.DataFrame({
    "Validation": ["Random Split", "Grouped Split"],
    "MAE": [before_mae, after_mae],
    "MSE": [before_mse, after_mse],
    "R²": [before_r2, after_r2]
})

results

,Validation,MAE,MSE,R²
0,Random Split,0.018672,0.004277,-0.049820
1,Grouped Split,0.018260,0.006019,-0.063314


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*



I reviewed all features used in the final model.

Identifier columns (`client_hash_id` and `content_hash_id`) were excluded because they identify records rather than describe page performance.

The target variable (CTR) was not used as an input feature.

Columns used to calculate the target, such as `gsc_clicks`, were also excluded to avoid target leakage.

The selected features represent information that would be available before making a prediction.

In [32]:
audit = pd.DataFrame({
    "Feature": [
        "gsc_impressions",
        "gsc_avg_position",
        "ga4_pageviews",
        "ga4_sessions",
        "scroll_events",
        "gsc_clicks",
        "CTR"
    ],
    "Used in Model": [
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "No",
        "No"
    ],
    "Leakage Risk": [
        "Low",
        "Low",
        "Low",
        "Low",
        "Low",
        "High",
        "High"
    ]
})

audit

,Feature,Used in Model,Leakage Risk
0,gsc_impressions,Yes,Low
1,gsc_avg_position,Yes,Low
2,ga4_pageviews,Yes,Low
3,ga4_sessions,Yes,Low
4,scroll_events,Yes,Low
5,gsc_clicks,No,High
6,CTR,No,High


Error Analysis

The largest errors occurred on pages with unusually high or unusually low CTR values. These cases may represent behaviour that is difficult to predict using the selected features alone.

In [34]:
comparison = pd.DataFrame({
    "Actual CTR": y_test,
    "Predicted CTR": pred
})

comparison["Absolute Error"] = (
    comparison["Actual CTR"] - comparison["Predicted CTR"]
).abs()

comparison.sort_values(
    "Absolute Error",
    ascending=False
).head(10)

,Actual CTR,Predicted CTR,Absolute Error
6192,1.0,0.000000,1.000000
6871,1.0,0.000000,1.000000
8699,1.0,0.000000,1.000000
4260,1.0,0.000000,1.000000
4261,1.0,0.000000,1.000000
8979,1.0,0.000000,1.000000
4121,1.0,0.000000,1.000000
6221,1.0,0.035193,0.964807
6874,1.0,0.035624,0.964376
6658,1.0,0.162767,0.837233


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

The largest errors occurred on pages with unusually high or unusually low CTR values. These cases may represent behaviour that is difficult to predict using the selected features alone.



### Original claim

The Random Forest model accurately predicts CTR.

### Revised claim

The Random Forest model showed measurable predictive ability on the evaluation dataset.

The grouped validation suggests that the model may generalise to unseen clients, although additional evaluation on larger datasets would strengthen this conclusion.

The model is intended as a decision-support tool rather than a fully automated prediction system.

In [35]:
print("Claim rewritten using safe language.")

Claim rewritten using safe language.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.